# ==============================================================================
# 💧 ENVIRONMENTAL HEALTH & SAFETY: WATER POTABILITY VIA GAUSSIAN NAIVE BAYES
# ==============================================================================
### Mathematical Formulation:
Gaussian Naive Bayes models the conditional probability of continuous chemical constituents:
$$P(x_i \mid y = c) = \frac{1}{\sqrt{2\pi \sigma_{ic}^2}} \exp\left( -\frac{(x_i - \mu_{ic})^2}{2\sigma_{ic}^2} \right)$$

In water chemistry, metrics like Solids, Chloramines, and Sulfates can span wide dynamic ranges.
To prevent zero-variance singularities in low-sample sub-regions and optimize decision boundaries,
we tune the variance smoothing regularization parameter:
$$\sigma_{ic,\text{smoothed}}^2 = \sigma_{ic}^2 + \epsilon \cdot \max(\sigma^2)$$
where $\epsilon$ is `var_smoothing`.


In [1]:
# STEP 1: IMPORTS & ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Environmental ML Analytics environment ready.")


✅ STEP 1: Environmental ML Analytics environment ready.


## 📥 STEP 2 & 3: INGESTION & COLUMN SANITIZATION
Loading the Kaggle Water Potability dataset with 9 physical and chemical water quality indicators.


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/water_potability/water_potability.csv'
df = pd.read_csv(data_path)

# Standardize column naming
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

print(f"📊 Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"📋 Attributes: {list(df.columns)}")
df.head(3)


📊 Dataset Shape: 3,276 rows, 10 columns
📋 Attributes: ['ph', 'hardness', 'solids', 'chloramines', 'sulfate', 'conductivity', 'organic_carbon', 'trihalomethanes', 'turbidity', 'potability']


,ph,hardness,solids,chloramines,sulfate,conductivity,organic_carbon,trihalomethanes,turbidity,potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE VERIFICATION
Separating target `potability` and handling missing physicochemical readings.


In [3]:
# STEP 4 & 5: SEGREGATE & HYGIENE
X = df.drop(columns=['potability'])
y = df['potability']

print("Missing values per sensor reading:")
print(X.isnull().sum()[X.isnull().sum() > 0])
print(f"\nTarget Distribution (Potability):")
print(y.value_counts(normalize=True).round(4) * 100)


Missing values per sensor reading:
ph                 491
sulfate            781
trihalomethanes    162
dtype: int64

Target Distribution (Potability):
potability
0    60.99
1    39.01
Name: proportion, dtype: float64


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
Preserving potability class distribution in an 80/20 train/test partition.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training Instances : {X_train.shape[0]} (Potable: {y_train.mean()*100:.1f}%)")
print(f"Testing Instances  : {X_test.shape[0]} (Potable: {y_test.mean()*100:.1f}%)")


Training Instances : 2620 (Potable: 39.0%)
Testing Instances  : 656 (Potable: 39.0%)


## ⚙️ STEP 7: PREPROCESSING PIPELINE (IMPUTATION & GAUSSIAN NORMALIZATION)
SimpleImputer with median strategy handles water testing equipment dropouts,
followed by Yeo-Johnson PowerTransformer to enforce Gaussian distribution requirements.


In [5]:
# STEP 7: PREPROCESSING PIPELINE
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer(method='yeo-johnson'))
])
print("✅ STEP 7: Robust Imputation & Power Transformation configured.")


✅ STEP 7: Robust Imputation & Power Transformation configured.


## ⚠️ STEP 8: EXPERIMENTATION & RAW SENSITIVITY TRAP
Demonstrating why missing value imputation and power transformation are critical for GaussianNB.


In [6]:
# STEP 8: BASELINE PIPELINE
pipe_base = Pipeline([
    ('prep', preprocessor),
    ('gnb', GaussianNB())
])
pipe_base.fit(X_train, y_train)

y_pred_base = pipe_base.predict(X_test)
acc_base = accuracy_score(y_test, y_pred_base)
roc_base = roc_auc_score(y_test, pipe_base.predict_proba(X_test)[:, 1])

print(f"Base GaussianNB Potability Test Accuracy : {acc_base*100:.2f}%")
print(f"Base GaussianNB Potability ROC-AUC       : {roc_base*100:.2f}%")


Base GaussianNB Potability Test Accuracy : 61.28%
Base GaussianNB Potability ROC-AUC       : 60.94%


## 🎯 STEP 9: HYPERPARAMETER TUNING (`var_smoothing`)
Optimizing `var_smoothing` across logarithmic intervals with Stratified 5-Fold Cross Validation.


In [7]:
# STEP 9: TUNING VAR_SMOOTHING
param_grid = {
    'gnb__var_smoothing': np.logspace(-11, -1, 15)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipe_base,
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"🏆 Best var_smoothing parameter: {grid_search.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid_search.best_score_*100:.2f}%")


🏆 Best var_smoothing parameter: {'gnb__var_smoothing': np.float64(1e-11)}
🎯 Best 5-Fold CV ROC-AUC: 58.74%


## ⚔️ STEP 10: ALGORITHM DUEL (GaussianNB vs Logistic Regression)
Evaluating Gaussian Naive Bayes against Logistic Regression.


In [8]:
# STEP 10: MODEL DUEL
pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)
roc_lr = roc_auc_score(y_test, pipe_lr.predict_proba(X_test)[:, 1])

print("="*65)
print(f"GaussianNB (Tuned)  -> Test Accuracy: {accuracy_score(y_test, best_model.predict(X_test))*100:.2f}%, ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])*100:.2f}%")
print(f"Logistic Regression -> Test Accuracy: {accuracy_score(y_test, y_pred_lr)*100:.2f}%, ROC-AUC: {roc_lr*100:.2f}%")
print("="*65)


GaussianNB (Tuned)  -> Test Accuracy: 61.28%, ROC-AUC: 60.94%
Logistic Regression -> Test Accuracy: 60.98%, ROC-AUC: 55.06%


## 📊 STEP 11: WATER SAFETY DIAGNOSTICS & EVALUATION
Safety-critical diagnostic report and Confusion Matrix.


In [9]:
# STEP 11: EVALUATION METRICS
y_pred_best = best_model.predict(X_test)
print("📋 WATER SAFETY CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_best, target_names=['Non-Potable (Unsafe)', 'Potable (Safe)']))

cm = confusion_matrix(y_test, y_pred_best)
print(f"Confusion Matrix:\n{cm}")


📋 WATER SAFETY CLASSIFICATION REPORT:
                      precision    recall  f1-score   support

Non-Potable (Unsafe)       0.63      0.89      0.74       400
      Potable (Safe)       0.51      0.18      0.27       256

            accuracy                           0.61       656
           macro avg       0.57      0.54      0.50       656
        weighted avg       0.58      0.61      0.55       656

Confusion Matrix:
[[355  45]
 [209  47]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & IOT SENSOR SMOKE TEST
Serializing model pipeline and testing sub-millisecond IoT inference latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & LATENCY BENCHMARK
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/water_potability_gaussian_nb_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_water_sensor = pd.DataFrame([{
    'ph': 7.15,
    'hardness': 185.0,
    'solids': 20000.0,
    'chloramines': 6.8,
    'sulfate': 320.0,
    'conductivity': 450.0,
    'organic_carbon': 14.5,
    'trihalomethanes': 65.0,
    'turbidity': 3.5
}])

t0 = time.perf_counter()
pred_potability = loaded_pipe.predict(sample_water_sensor)[0]
pred_potability_prob = loaded_pipe.predict_proba(sample_water_sensor)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n💧 LIVE IOT SENSOR POTABILITY AUDIT:")
print(f"   Safety Status : {'✅ POTABLE (SAFE TO DRINK)' if pred_potability == 1 else '🚨 NON-POTABLE (UNSAFE)'}")
print(f"   Potability Probability : {pred_potability_prob*100:.2f}%")
print(f"   Sensor Inference Latency: {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/water_potability_gaussian_nb_pipeline.joblib (2,722 bytes)

💧 LIVE IOT SENSOR POTABILITY AUDIT:
   Safety Status : 🚨 NON-POTABLE (UNSAFE)
   Potability Probability : 28.55%
   Sensor Inference Latency: 14.302 ms (SLA < 2.0ms: CHECK)
